# Processing-variant comparison — Sentinel-1 ocean current retrieval

Side-by-side comparison of the retrieved radial ocean current (`v_current_ocn`)
against the **GLO12** model and the **OCN** product, across several processing
chains and block sizes — all for **scene1, IW1, VV**.

**Variants**
- **OCN product** — current derived directly from the L2 OCN `rvlDcObs`.
- **GAMMA mosaic-first** — `deramp → SLC_mosaic_ScanSAR → doppler_2d_SLC` on the
  mosaic (the current `shell_commands` order), at block sizes 256 / 512 / 1024.
- **GAMMA mosaic-last** — `deramp → doppler_2d_SLC per burst → stitch`: the
  Doppler centroid is estimated **before** any mosaic, then the per-burst grids
  are concatenated in azimuth. Same block sizes.
- **Our pipeline, `merge_first=True`** — bursts merged into one continuous image,
  then Doppler estimation, at azimuth block sizes 64 / 128 / 256.
- **Our pipeline, `merge_first=False`** — per-burst Doppler estimation, same sizes.

> `doppler_2d_SLC` enforces a minimum block size of **256 lines**, so the GAMMA
> sweep starts at 256. Our own estimator has no such limit, so it sweeps 64–256.

**Prerequisites** — the GAMMA variants reuse the deramped mosaic / burst SLCs
produced by `shell_commands/gamma_iw1_deramp_merge_doppler.sh`. Run that script
once first if `data/sentinel-1/gamma_iw1/` is empty.

**Caching** — heavy Python-pipeline results are pickled under
`data/sentinel-1/scene1/comparison_cache/` and GAMMA Doppler `.npz` files under
`data/sentinel-1/gamma_iw1/`. The first run is slow (tens of minutes); re-runs
are near-instant. Delete `comparison_cache/` to force a recompute.

In [ ]:
import os, sys

# Locate the repo root (the directory containing `scripts/`) and make it importable.
_root = os.getcwd()
while _root != os.path.dirname(_root) and not os.path.isdir(os.path.join(_root, 'scripts')):
    _root = os.path.dirname(_root)
if not os.path.isdir(os.path.join(_root, 'scripts')):
    raise RuntimeError('Could not locate repo root (no scripts/ directory found)')
os.chdir(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
print('repo root:', _root)

import pickle
import numpy as np
import matplotlib.pyplot as plt

from scripts.sentinel_1.pipeline import run_all_bursts, run_gamma_dop2d_pipeline
from scripts.sentinel_1.gamma_variants import (
    gamma_doppler_mosaic_first, gamma_doppler_mosaic_last,
)
from scripts.sentinel_1.grid_merge import (
    merge_burst_grids, merge_model_grid, compute_stats,
)

## Data paths

In [ ]:
import glob

DATA  = 'data'
SCENE = 'scene1'
S1_DIR = f'{DATA}/sentinel-1/{SCENE}'

SLC_SAFE  = f'{S1_DIR}/S1A_IW_SLC.SAFE'
OCN_SAFE  = f'{S1_DIR}/S1A_IW_OCN.SAFE'
POEORB    = glob.glob(f'{S1_DIR}/S1A_OPER_AUX_POEORB_*.EOF')[0]
AUX_CAL   = f'{DATA}/sentinel-1/S1A_AUX_CAL_V20190228T092500_G20240327T102320.SAFE'
ANNOT_XML = glob.glob(f'{SLC_SAFE}/annotation/s1a-iw1-slc-vv-*.xml')[0]

ERA5_WIND = f'{DATA}/era5_data/{SCENE}/era5_wind.nc'
ERA5_WAVE = f'{DATA}/era5_data/{SCENE}/era5_wave.nc'
GLO12     = f'{DATA}/era5_data/{SCENE}/glo12.nc'

SUBSWATH, POL = 'iw1', 'vv'

_paths = [SLC_SAFE, OCN_SAFE, POEORB, AUX_CAL, ANNOT_XML, ERA5_WIND, ERA5_WAVE, GLO12]
print('all paths exist:', all(os.path.exists(p) for p in _paths))
for p in _paths:
    if not os.path.exists(p):
        print('  MISSING:', p)

## Configuration — block-size sweeps and caching

In [ ]:
# Block-size sweeps.
GAMMA_BLSZ    = [256, 512, 1024]   # doppler_2d_SLC azimuth block (GAMMA minimum 256)
OURS_BLOCK_AZ = [64, 128, 256]     # our estimator azimuth block (no minimum)

# Fixed range-block settings for our pipeline.
OURS_BLOCK_RG, OURS_STRIDE_RG = 256, 128

# Pickle cache for the (slow) Python-pipeline results.
CACHE = f'{S1_DIR}/comparison_cache'
os.makedirs(CACHE, exist_ok=True)

def cached(key, fn):
    """Pickle-cache the return value of fn() under CACHE/key.pkl."""
    path = f'{CACHE}/{key}.pkl'
    if os.path.exists(path):
        print(f'[cache]   {key}')
        with open(path, 'rb') as f:
            return pickle.load(f)
    print(f'[compute] {key} ...')
    obj = fn()
    with open(path, 'wb') as f:
        pickle.dump(obj, f)
    return obj

def as_list(result):
    """Normalise a pipeline result (single dict or list) to list[dict]."""
    return result if isinstance(result, list) else [result]

## 1 — OCN product

Current derived directly from the L2 OCN `rvlDcObs` (`use_ocn_dc=True`); no SLC
reading or Doppler estimation — it stays on the OCN output grid.

In [ ]:
ocn_result = cached('ocn_product', lambda: run_all_bursts(
    SLC_SAFE, SUBSWATH, POEORB, AUX_CAL, OCN_SAFE,
    ERA5_WIND, ERA5_WAVE, GLO12, POL, use_ocn_dc=True,
))
print('OCN product:', len(as_list(ocn_result)), 'grid(s)')

## 2 — GAMMA: mosaic-first vs mosaic-last

**mosaic-first** re-runs `doppler_2d_SLC` on the existing deramped mosaic SLC.

**mosaic-last** runs `doppler_2d_SLC` once per deramped burst (over that burst's
valid-line range, so no block straddles a burst boundary) and stitches the
per-burst Doppler grids in azimuth. GAMMA's `SLC_mosaic_ScanSAR` only mosaics
complex SLC data, not Doppler grids, so the final stitch is a plain azimuth
concatenation.

Both write a `.npz` with the shell-script schema, then go through
`run_gamma_dop2d_pipeline` for the geometry / Stokes / wave / mispointing
corrections and the GLO12 comparison.

In [ ]:
def run_gamma_variant(mosaic_mode, blsz):
    """Generate (or load) the GAMMA Doppler .npz, then apply the corrections pipeline."""
    if mosaic_mode == 'mosaic_first':
        npz = gamma_doppler_mosaic_first(blsz)
    else:
        npz = gamma_doppler_mosaic_last(blsz)
    return run_gamma_dop2d_pipeline(
        dop2d_npz=npz, annotation_xml=ANNOT_XML, subswath=SUBSWATH,
        poeorb_path=POEORB, aux_cal_path=AUX_CAL, ocn_safe=OCN_SAFE,
        era5_wind=ERA5_WIND, era5_wave=ERA5_WAVE, glo12=GLO12, polarisation=POL,
    )

gamma_first = {
    b: cached(f'gamma_mosaic_first_blsz{b}',
              lambda b=b: run_gamma_variant('mosaic_first', b))
    for b in GAMMA_BLSZ
}
gamma_last = {
    b: cached(f'gamma_mosaic_last_blsz{b}',
              lambda b=b: run_gamma_variant('mosaic_last', b))
    for b in GAMMA_BLSZ
}

In [ ]:
GAMMA_VARIANTS = []
for b in GAMMA_BLSZ:
    GAMMA_VARIANTS.append((f'GAMMA mosaic-first  blsz={b}', as_list(gamma_first[b])))
for b in GAMMA_BLSZ:
    GAMMA_VARIANTS.append((f'GAMMA mosaic-last  blsz={b}', as_list(gamma_last[b])))

print(f'{len(GAMMA_VARIANTS)} GAMMA variants assembled')


def gamma_variant_grid(results):
    """Merge one GAMMA variant onto a regular grid and score it vs GLO12."""
    glat, glon, sar = merge_burst_grids(results, variable='v_current', overlap='average')
    _, _, model = merge_model_grid(results)
    bias, rmse, r = compute_stats(sar, model)
    return dict(glat=glat, glon=glon, sar=sar, model=model, bias=bias, rmse=rmse, r=r)


def ocn_reference_grid(results):
    """Merge the OCN product onto a regular grid."""
    glat, glon, sar = merge_burst_grids(results, variable='v_current_ocn', overlap='average')
    return dict(glat=glat, glon=glon, sar=sar)


GRIDS = {label: gamma_variant_grid(res) for label, res in GAMMA_VARIANTS}
labels = [label for label, _ in GAMMA_VARIANTS]
OCN_REF = ocn_reference_grid(as_list(ocn_result))

print('done merging GAMMA.')

# Common colour scale across OCN + all GAMMA panels.
_all = [OCN_REF['sar'][np.isfinite(OCN_REF['sar'])]]
for g in GRIDS.values():
    _all.append(g['sar'][np.isfinite(g['sar'])])
_all = np.concatenate(_all)
VMAX = float(np.nanpercentile(np.abs(_all), 98))

n_panels = len(labels) + 1
ncols = 4
nrows = int(np.ceil(n_panels / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(4.6 * ncols, 4.0 * nrows))
axes = axes.ravel()

# Panel 0: OCN product reference
ext_ref = [
    OCN_REF['glon'].min(), OCN_REF['glon'].max(),
    OCN_REF['glat'].min(), OCN_REF['glat'].max()
]
im = axes[0].imshow(
    OCN_REF['sar'],
    extent=ext_ref,
    origin='lower',
    cmap='RdBu_r',
    vmin=-VMAX,
    vmax=VMAX,
    aspect='auto'
)
axes[0].set_title('OCN product (reference)', fontweight='bold')
axes[0].set_xlabel('Longitude [deg]')
axes[0].set_ylabel('Latitude [deg]')

for ax, label in zip(axes[1:], labels):
    g = GRIDS[label]
    ext = [g['glon'].min(), g['glon'].max(), g['glat'].min(), g['glat'].max()]
    ax.imshow(
        g['sar'],
        extent=ext,
        origin='lower',
        cmap='RdBu_r',
        vmin=-VMAX,
        vmax=VMAX,
        aspect='auto'
    )
    ax.set_title(
        f"{label}\nbias={g['bias']:+.3f}  RMSE={g['rmse']:.3f}  r={g['r']:.3f}",
        fontsize=9
    )
    ax.set_xlabel('Longitude [deg]')
    ax.set_ylabel('Latitude [deg]')

for ax in axes[n_panels:]:
    ax.axis('off')

fig.colorbar(
    im,
    ax=axes.tolist(),
    label='radial current [m/s]',
    fraction=0.015,
    pad=0.02
)
fig.suptitle(
    f'GAMMA retrieved radial ocean current vs OCN product — {SCENE} {SUBSWATH.upper()}  '
    f'(colour scale +/-{VMAX:.2f} m/s)',
    fontsize=13,
    y=1.005
)

plt.tight_layout()
plt.show()

## 3 — Our pipeline: `merge_first` True / False

`merge_first=True` deramps and coherently merges all bursts into one continuous
image, then estimates the Doppler centroid once. `merge_first=False` estimates
per burst. Each is swept over azimuth block sizes 64 / 128 / 256 (the stride is
half the block; range blocking is held fixed).

In [ ]:
def run_ours(merge_first, block_az):
    return run_all_bursts(
        SLC_SAFE, SUBSWATH, POEORB, AUX_CAL, OCN_SAFE,
        ERA5_WIND, ERA5_WAVE, GLO12, POL,
        merge_first=merge_first, deramp_method='esa_eq1', use_ocn_dc=False,
        block_az=block_az, stride_az=max(block_az // 2, 1),
        block_rg=OURS_BLOCK_RG, stride_rg=OURS_STRIDE_RG,
    )

ours_merged = {
    b: cached(f'ours_merge_first_blsz{b}', lambda b=b: run_ours(True, b))
    for b in OURS_BLOCK_AZ
}
ours_perburst = {
    b: cached(f'ours_perburst_blsz{b}', lambda b=b: run_ours(False, b))
    for b in OURS_BLOCK_AZ
}

## 4 — Assemble all variants

Each result is normalised to `list[dict]`, merged onto a regular lat/lon grid
via `merge_burst_grids`, and scored against GLO12.

In [ ]:
# Ordered list of (label, results), results normalised to list[dict].
VARIANTS = [('OCN product', as_list(ocn_result))]
for b in GAMMA_BLSZ:
    VARIANTS.append((f'GAMMA mosaic-first  blsz={b}', as_list(gamma_first[b])))
for b in GAMMA_BLSZ:
    VARIANTS.append((f'GAMMA mosaic-last  blsz={b}', as_list(gamma_last[b])))
for b in OURS_BLOCK_AZ:
    VARIANTS.append((f'ours merge_first=True  block_az={b}', as_list(ours_merged[b])))
for b in OURS_BLOCK_AZ:
    VARIANTS.append((f'ours merge_first=False  block_az={b}', as_list(ours_perburst[b])))

print(f'{len(VARIANTS)} variants assembled')

def variant_grid(results):
    """Merge a variant onto a regular grid and score it vs GLO12."""
    glat, glon, sar = merge_burst_grids(results, variable='v_current_ocn', overlap='average')
    _, _, model = merge_model_grid(results)
    bias, rmse, r = compute_stats(sar, model)
    return dict(glat=glat, glon=glon, sar=sar, model=model, bias=bias, rmse=rmse, r=r)

GRIDS  = {label: variant_grid(res) for label, res in VARIANTS}
labels = [label for label, _ in VARIANTS]
print('done merging.')

## 5 — Side-by-side current maps

Every panel shares one colour scale. Panel 0 is the GLO12 model; the rest are
the retrieved SAR current for each variant, titled with bias / RMSE / r vs GLO12.

In [ ]:
# Common colour scale across every panel (98th percentile of all SAR + model).
_all = []
for g in GRIDS.values():
    _all.append(g['sar'][np.isfinite(g['sar'])])
    _all.append(g['model'][np.isfinite(g['model'])])
_all = np.concatenate(_all)
VMAX = float(np.nanpercentile(np.abs(_all), 98))

n_panels = len(labels) + 1          # + GLO12 reference
ncols = 4
nrows = int(np.ceil(n_panels / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(4.6 * ncols, 4.0 * nrows))
axes = axes.ravel()

# Panel 0: GLO12 reference (widest-coverage variant's model grid).
ref = max(GRIDS.values(), key=lambda g: int(np.isfinite(g['model']).sum()))
ext_ref = [ref['glon'].min(), ref['glon'].max(), ref['glat'].min(), ref['glat'].max()]
im = axes[0].imshow(ref['model'], extent=ext_ref, origin='lower',
                    cmap='RdBu_r', vmin=-VMAX, vmax=VMAX, aspect='auto')
axes[0].set_title('GLO12 model (reference)', fontweight='bold')
axes[0].set_xlabel('Longitude [deg]'); axes[0].set_ylabel('Latitude [deg]')

for ax, label in zip(axes[1:], labels):
    g = GRIDS[label]
    ext = [g['glon'].min(), g['glon'].max(), g['glat'].min(), g['glat'].max()]
    ax.imshow(g['sar'], extent=ext, origin='lower',
              cmap='RdBu_r', vmin=-VMAX, vmax=VMAX, aspect='auto')
    ax.set_title(f"{label}\nbias={g['bias']:+.3f}  RMSE={g['rmse']:.3f}  r={g['r']:.3f}",
                 fontsize=9)
    ax.set_xlabel('Longitude [deg]'); ax.set_ylabel('Latitude [deg]')

for ax in axes[n_panels:]:
    ax.axis('off')

fig.colorbar(im, ax=axes.tolist(), label='radial current [m/s]',
             fraction=0.015, pad=0.02)
fig.suptitle(f'Retrieved radial ocean current vs GLO12 — {SCENE} {SUBSWATH.upper()}  '
             f'(colour scale +/-{VMAX:.2f} m/s)', fontsize=13, y=1.005)

plt.tight_layout()
plt.show()

## 6 — Summary statistics

In [ ]:
rows = [(lbl, GRIDS[lbl]['bias'], GRIDS[lbl]['rmse'], GRIDS[lbl]['r']) for lbl in labels]

print(f"{'variant':<40}{'bias':>10}{'RMSE':>10}{'r':>9}")
print('-' * 69)
for lbl, bias, rmse, r in rows:
    print(f'{lbl:<40}{bias:>+10.4f}{rmse:>10.4f}{r:>9.4f}')

try:
    import pandas as pd
    stats_df = pd.DataFrame(rows, columns=['variant', 'bias', 'RMSE', 'r']).set_index('variant')
except ImportError:
    stats_df = None
stats_df

In [ ]:
# RMSE and correlation, ranked.
fig, (axr, axc) = plt.subplots(1, 2, figsize=(15, max(4.0, 0.45 * len(labels))))
ypos = np.arange(len(labels))

axr.barh(ypos, [GRIDS[l]['rmse'] for l in labels], color='tab:red', alpha=0.85)
axr.set_yticks(ypos); axr.set_yticklabels(labels, fontsize=8)
axr.invert_yaxis(); axr.set_xlabel('RMSE vs GLO12 [m/s]')
axr.set_title('RMSE  (lower is better)'); axr.grid(axis='x', alpha=0.3)

axc.barh(ypos, [GRIDS[l]['r'] for l in labels], color='tab:blue', alpha=0.85)
axc.set_yticks(ypos); axc.set_yticklabels([]); axc.invert_yaxis()
axc.set_xlabel('correlation r vs GLO12')
axc.set_title('correlation  (higher is better)'); axc.grid(axis='x', alpha=0.3)
axc.axvline(0, color='k', lw=0.8)

plt.tight_layout(); plt.show()

## 7 — Scatter against GLO12

Per-variant scatter of the retrieved SAR current against GLO12, with the 1:1
line. Tighter clustering around the diagonal means a better match.

In [ ]:
fig, axes = plt.subplots(nrows, ncols, figsize=(3.6 * ncols, 3.4 * nrows))
axes = axes.ravel()
lim = VMAX * 1.05

for ax, label in zip(axes, labels):
    g = GRIDS[label]
    m = np.isfinite(g['sar']) & np.isfinite(g['model'])
    ax.scatter(g['model'][m], g['sar'][m], s=3, alpha=0.25, rasterized=True)
    ax.plot([-lim, lim], [-lim, lim], 'k--', lw=1)
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim); ax.set_aspect('equal')
    ax.set_title(f"{label}\nRMSE={g['rmse']:.3f}  r={g['r']:.3f}", fontsize=8)
    ax.set_xlabel('GLO12 [m/s]'); ax.set_ylabel('SAR [m/s]')

for ax in axes[len(labels):]:
    ax.axis('off')

fig.suptitle('Retrieved current vs GLO12 — per-variant scatter', fontsize=13, y=1.005)
plt.tight_layout(); plt.show()